# Módulo 8: Seguridad con agentes — Análisis Autónomo de Amenazas (20 min)

En este módulo vas a construir con **Strands Agents** una versión mínima y didáctica de **ATA (Autonomous Threat Analysis)**, el sistema que Amazon usa para anticipar amenazas con agentes de IA que compiten entre sí: un **equipo rojo** que simula a un atacante y un **equipo azul** que mejora las reglas de detección.

Basado en el artículo de Amazon Science *"How Amazon uses AI agents to anticipate and counter cyber threats"* (Daniel Weiss, Amazon Security).

**Prerrequisitos:** Módulos 1 (agentes y herramientas), 2 (hooks) y 6 (multi-agente). Credenciales de AWS configuradas (Strands usa Amazon Bedrock por defecto).

> ⚠️ **Todo es simulado.** No se ejecuta ningún comando real ni se toca ningún sistema. El "entorno aislado" es una lista de Python en memoria y las "técnicas" son eventos de telemetría sintética. El objetivo es aprender los **patrones de arquitectura** de un sistema defensivo con agentes.

---

## Contexto: ¿qué es ATA?

**El problema.** Los atacantes usan cada vez más IA y cambian sus técnicas muy rápido. Probar a mano si nuestras reglas de detección cubren cada variante lleva **semanas**.

**El origen.** ATA nació en un hackathon interno de Amazon en agosto de 2024. En solo 48 horas, un prototipo encontró un hueco en una regla de detección y generó automáticamente una versión mejorada.

**Cómo funciona.**

- **Equipo rojo (agentes atacantes):** simulan las técnicas de un adversario.
- **Equipo azul (agentes defensores):** comprueban si las detecciones cubren esas técnicas y crean reglas nuevas o mejoradas.
- **Flujo en grafo:** cada nodo es un agente especializado y la salida de uno es la entrada del siguiente.
- **Entornos aislados:** las pruebas corren en réplicas de producción, sin datos de clientes y sin riesgo para la operación real.
- **Ejecución fundamentada (*grounded execution*):** ninguna afirmación del agente se acepta sin evidencia. Si un agente dice que ejecutó una técnica, tiene que haber un log con marca de tiempo de un host concreto que lo demuestre. Las métricas de detección (precisión y recall) se calculan consultando logs reales. Así se mitigan las **alucinaciones**.

**Caso de estudio del artículo.** Para las *reverse shells* en Python, el equipo rojo generó y ejecutó 37 variaciones exploratorias y luego 64 variantes dirigidas. El sistema creó una regla mejorada que alcanzó **precisión 1.00 y recall 1.00** frente a esas variantes y una hora de datos de auditoría de producción.

**Salvaguardas.** Entornos efímeros y aislados, controles de acceso estrictos, registro de auditoría completo y **aprobación humana obligatoria** antes de llevar cualquier cambio a producción.

**Impacto.** Cuando una ejecución falla, los agentes analizan el error y lo reintentan (normalmente aciertan en menos de 3 intentos). El ciclo completo pasa de semanas a unas **4 horas (−96 %)**, con 10 a 30 variaciones en paralelo. La IA **aumenta** al equipo humano, no lo reemplaza.

### Dos métricas que vas a ver todo el rato

| Métrica | Pregunta que responde | Si es baja… |
|---|---|---|
| **Precisión** | De todas las alertas que lanzó la regla, ¿cuántas eran ataques de verdad? | Muchas **falsas alarmas** (el equipo de guardia deja de confiar en la regla) |
| **Recall** (exhaustividad) | De todos los ataques reales, ¿cuántos detectó la regla? | **Ataques sin detectar** (huecos en la defensa) |

El objetivo del equipo azul es subir ambas a 1.0 a la vez, y eso es más difícil de lo que parece: una regla muy amplia detecta todo, pero también salta con la actividad legítima.

### Qué vamos a construir

```bash
                ┌─────────────┐  evidencias  ┌─────────────┐  regla validada  ┌──────────┐
  tarea ─────▶  │ equipo_rojo │ ──────────▶ │ equipo_azul │ ──────────────▶ │ revisor  │ ──▶ solicitud de cambio
                └─────────────┘              └─────────────┘  (P = 1, R = 1)  └──────────┘          │
                       ▲                            │                                               ▼
                       └──── la regla aún falla ────┘                                 👤 aprobación humana ──▶ producción
```

| Concepto de ATA | Cómo lo implementamos con Strands |
|---|---|
| Equipo rojo / equipo azul | Dos `Agent` con herramientas (`@tool`) distintas |
| Flujo en grafo | `GraphBuilder` con aristas condicionales (ciclo rojo ↔ azul) |
| Ejecución fundamentada | Las herramientas escriben y leen evidencia en una "base de logs"; el grafo decide con métricas calculadas en Python, no con la opinión del LLM |
| Entorno aislado | Hook `BeforeToolCallEvent` que bloquea cualquier host fuera del sandbox |
| Auditoría | Hook `AfterToolCallEvent` que registra cada llamada a herramienta |
| Supervisión humana | Interrupción (`event.interrupt`) antes de desplegar en producción |
| Refinamiento ante errores | El propio bucle del agente: lee el error de la herramienta y reintenta |

---

## Parte 1: El entorno aislado (sandbox simulado)

Primero creamos, en Python puro y sin LLM, todo lo que en ATA es "infraestructura real":

- **`SANDBOX`**: los hosts del entorno aislado y los intérpretes instalados en cada uno.
- **`LOGS`**: la base de datos de telemetría. Cada evento lleva un ID (`EVT-0001`…), que es la **evidencia**.
- **Línea base**: actividad legítima que equivale a la "hora de datos de auditoría de producción" del artículo. Sirve para medir las falsas alarmas.

In [1]:
import json
from datetime import datetime, timezone
from IPython.display import Markdown, display

# Hosts del entorno aislado y los intérpretes instalados en cada uno
SANDBOX = {
    "sandbox-01": ["python3", "python3.12"],
    "sandbox-02": ["python", "python3"],
    "sandbox-03": ["python3.11"],
}

# La "base de datos de logs": cada evento es telemetría de un proceso
LOGS: list[dict] = []


def log(mensaje: str) -> None:
    """Imprime una línea completa de una vez (las herramientas pueden ejecutarse en paralelo)."""
    print(mensaje + "\n", end="", flush=True)


def registrar_evento(host, proceso, proceso_padre, puerto_destino, destino_externo, stdin_a_socket, origen):
    """Guarda un evento en la base de logs y devuelve su ID (la evidencia)."""
    evento = {
        "id": f"EVT-{len(LOGS) + 1:04d}",
        "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "host": host,
        "proceso": proceso,
        "proceso_padre": proceso_padre,
        "puerto_destino": puerto_destino,
        "destino_externo": destino_externo,   # ¿la conexión sale de la red interna?
        "stdin_a_socket": stdin_a_socket,     # ¿la entrada del proceso está conectada a la red?
        "origen": origen,                     # "linea_base" (legítimo) o "equipo_rojo" (simulación)
    }
    LOGS.append(evento)
    return evento["id"]


# Línea base: actividad legítima
LINEA_BASE = [
    # host,        proceso,      padre,     puerto, externo, stdin_a_socket
    ("sandbox-01", "python3",    "cron",    443,    True,    False),  # script que llama a una API
    ("sandbox-01", "python3",    "bash",    443,    True,    False),  # pip install
    ("sandbox-02", "python",     "systemd", 5432,   False,   False),  # app que consulta su base de datos
    ("sandbox-02", "python3",    "jupyter", 8888,   False,   False),  # notebook local
    ("sandbox-03", "python3.11", "cron",    80,     True,    False),  # descarga de un informe
    ("sandbox-03", "ssh",        "bash",    22,     True,    True),   # sesión SSH de un administrador
    ("sandbox-01", "sshd",       "systemd", 22,     False,   True),   # servidor SSH
]
for host, proceso, padre, puerto, externo, stdin in LINEA_BASE:
    registrar_evento(host, proceso, padre, puerto, externo, stdin, origen="linea_base")

print(f"✅ Sandbox listo: {len(SANDBOX)} hosts, {len(LOGS)} eventos de línea base")

✅ Sandbox listo: 3 hosts, 7 eventos de línea base


### La regla de detección vigente (y su hueco)

Una regla es un conjunto de condiciones que deben cumplirse **todas a la vez** para lanzar una alerta. La regla actual (`v1`) se escribió pensando en el ejemplo "de manual": el proceso se llama exactamente `python` y conecta al puerto `4444`.

¿Qué pasa si el atacante usa `python3` o el puerto `443`? Eso es lo que el equipo rojo va a averiguar.

`calcular_metricas` es la pieza clave de la **ejecución fundamentada**: mide cualquier regla contra los eventos que **realmente** están en la base de logs.

In [2]:
# Regla vigente (v1): solo mira el nombre exacto "python" y el puerto 4444
REGLA_ACTUAL = {
    "nombre": "canal-remoto-python-v1",
    "procesos": ["python"],
    "puertos": [4444],
    "requiere_destino_externo": False,
    "requiere_stdin_a_socket": False,
}


def regla_detecta(regla: dict, evento: dict) -> bool:
    """True si el evento cumple TODAS las condiciones de la regla."""
    if evento["proceso"] not in regla["procesos"]:
        return False
    if regla["puertos"] and evento["puerto_destino"] not in regla["puertos"]:  # lista vacía = cualquier puerto
        return False
    if regla["requiere_destino_externo"] and not evento["destino_externo"]:
        return False
    if regla["requiere_stdin_a_socket"] and not evento["stdin_a_socket"]:
        return False
    return True


def calcular_metricas(regla: dict) -> dict:
    """Mide la regla contra todos los eventos de la base de logs."""
    vp = fp = fn = 0  # verdaderos positivos, falsos positivos, falsos negativos
    for evento in LOGS:
        alerta = regla_detecta(regla, evento)
        es_ataque = evento["origen"] == "equipo_rojo"
        if alerta and es_ataque:
            vp += 1
        elif alerta:
            fp += 1
        elif es_ataque:
            fn += 1
    precision = vp / (vp + fp) if vp + fp else 0.0
    recall = vp / (vp + fn) if vp + fn else 0.0
    return {"precision": round(precision, 2), "recall": round(recall, 2),
            "verdaderos_positivos": vp, "falsos_positivos": fp, "falsos_negativos": fn}


print("Métricas de la regla v1:", calcular_metricas(REGLA_ACTUAL))

Métricas de la regla v1: {'precision': 0.0, 'recall': 0.0, 'verdaderos_positivos': 0, 'falsos_positivos': 0, 'falsos_negativos': 0}


Todavía no hay ningún ataque simulado en los logs, así que el recall es 0: **no podemos saber si la regla funciona hasta que alguien la ponga a prueba**. Ese es el trabajo del equipo rojo.

---

## Parte 2: Herramientas y salvaguardas

### Herramienta del equipo rojo

`ejecutar_variante_simulada` "ejecuta" una variante de la técnica dentro del sandbox. En realidad solo escribe un evento de telemetría en `LOGS` y devuelve su ID de evidencia.

Si el intérprete no está instalado en ese host, la herramienta lanza un `ValueError`. Strands no rompe el bucle: le devuelve al agente un resultado con estado `error` y el mensaje. El agente tendrá que leerlo y corregirse, igual que los agentes de ATA, que refinan su enfoque cuando una ejecución falla.

In [3]:
from strands import Agent, tool


@tool
def ejecutar_variante_simulada(host: str, proceso: str, proceso_padre: str, puerto_destino: int) -> str:
    """Ejecuta (de forma simulada) una variante de la técnica 'canal de control remoto
    desde un intérprete' en un host del entorno aislado y deja evidencia en los logs.

    Args:
        host: Host del entorno aislado donde se ejecuta la variante
        proceso: Nombre del intérprete que se usa (por ejemplo "python3")
        proceso_padre: Proceso que lanza al intérprete (por ejemplo "bash", "cron" o "nginx")
        puerto_destino: Puerto de destino de la conexión saliente
    """
    # El hook SoloSandboxHook garantiza que `host` pertenece al entorno aislado
    instalados = SANDBOX[host]
    if proceso not in instalados:
        log(f"[ROJO] ⚠️  {proceso} no está instalado en {host} → el agente tendrá que corregirse")
        raise ValueError(f"'{proceso}' no está instalado en {host}. Intérpretes disponibles: {instalados}")
    if not 1 <= puerto_destino <= 65535:
        raise ValueError(f"El puerto {puerto_destino} no es válido (1-65535)")

    evidencia = registrar_evento(host, proceso, proceso_padre, puerto_destino,
                                 destino_externo=True, stdin_a_socket=True, origen="equipo_rojo")
    log(f"[ROJO] 🔴 {proceso_padre} → {proceso} → puerto {puerto_destino} en {host} | evidencia {evidencia}")
    return f"Variante ejecutada en {host}. Evidencia: {evidencia}"


print("✅ Herramienta del equipo rojo definida")

✅ Herramienta del equipo rojo definida


### Herramientas del equipo azul

- `consultar_logs`: lee la base de logs, como el equipo azul de ATA consulta las bases de telemetría reales.
- `evaluar_regla`: mide una regla con `calcular_metricas`. **El agente no puede inventarse las métricas**: siempre salen de los datos.

`mejor_regla()` vuelve a medir todas las reglas probadas contra los logs **actuales** y devuelve la mejor. La usará el grafo para decidir si la regla está validada.

In [ ]:
REGLAS_PROBADAS: list[dict] = []


@tool
def consultar_logs() -> str:
    """Devuelve todos los eventos de la base de logs del entorno aislado en formato JSON."""
    log(f"[AZUL] 🔵 Consultando {len(LOGS)} eventos de la base de logs")
    return json.dumps(LOGS, ensure_ascii=False)


@tool
def evaluar_regla(nombre: str, procesos: list[str], puertos: list[int],
                  requiere_destino_externo: bool, requiere_stdin_a_socket: bool) -> str:
    """Mide la precisión y el recall de una regla de detección contra los logs reales.

    Args:
        nombre: Nombre de la regla (por ejemplo "canal-remoto-python-v2")
        procesos: Nombres exactos de proceso que vigila la regla
        puertos: Puertos de destino que vigila. Lista vacía = cualquier puerto
        requiere_destino_externo: Si es True, solo alerta cuando el destino está fuera de la red interna
        requiere_stdin_a_socket: Si es True, solo alerta cuando la entrada del proceso está conectada a la red
    """
    regla = {"nombre": nombre, "procesos": procesos, "puertos": puertos,
             "requiere_destino_externo": requiere_destino_externo,
             "requiere_stdin_a_socket": requiere_stdin_a_socket}
    REGLAS_PROBADAS.append(regla)
    metricas = calcular_metricas(regla)
    log(f"[AZUL] 🔵 {nombre}: precisión={metricas['precision']} recall={metricas['recall']}")
    return json.dumps(metricas)


def mejor_regla() -> dict | None:
    """Recalcula todas las reglas probadas con los logs actuales y devuelve la mejor."""
    candidatas = [{"regla": r, **calcular_metricas(r)} for r in REGLAS_PROBADAS]
    return max(candidatas, key=lambda c: c["precision"] + c["recall"], default=None)


print("✅ Herramientas del equipo azul definidas")

### Salvaguardas con hooks

Como viste en el Módulo 2, un **hook** es código determinista que se ejecuta en puntos concretos del bucle del agente. Aquí lo usamos para dos salvaguardas del artículo:

1. **`SoloSandboxHook`** (`BeforeToolCallEvent`): antes de ejecutar *cualquier* herramienta que reciba un `host`, comprueba que ese host pertenece al entorno aislado. Si no, cancela la llamada con `event.cancel_tool`. La política está en un único sitio y no depende de que el LLM "se porte bien".
2. **`AuditoriaHook`** (`AfterToolCallEvent`): registra cada llamada a herramienta (agente, herramienta, entrada y resultado) para tener trazabilidad completa.

In [ ]:
from strands.hooks import HookProvider, HookRegistry, BeforeToolCallEvent, AfterToolCallEvent


class SoloSandboxHook(HookProvider):
    """Salvaguarda: ninguna herramienta puede actuar sobre hosts fuera del entorno aislado."""

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeToolCallEvent, self.verificar_host)

    def verificar_host(self, event: BeforeToolCallEvent) -> None:
        host = event.tool_use["input"].get("host")
        if host is not None and host not in SANDBOX:
            log(f"[SALVAGUARDA] 🛑 Bloqueado: '{host}' no pertenece al entorno aislado")
            event.cancel_tool = f"Bloqueado por política: '{host}' está fuera del entorno aislado. Hosts permitidos: {list(SANDBOX)}"


AUDITORIA: list[dict] = []


class AuditoriaHook(HookProvider):
    """Registra cada llamada a herramienta: quién, qué, cuándo y con qué resultado."""

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(AfterToolCallEvent, self.registrar)

    def registrar(self, event: AfterToolCallEvent) -> None:
        AUDITORIA.append({
            "hora": datetime.now(timezone.utc).strftime("%H:%M:%S"),
            "agente": event.agent.name,
            "herramienta": event.tool_use["name"],
            "entrada": event.tool_use["input"],
            "estado": "cancelada" if event.cancel_message else event.result["status"],
        })


print("✅ Hooks de salvaguarda y auditoría definidos")

### Prueba la salvaguarda

Llamamos a la herramienta **directamente** (sin LLM) con un host de producción. Las llamadas directas (`agent.tool.<nombre>(...)`) también pasan por los hooks, así que la política se aplica igual.

In [ ]:
agente_prueba = Agent(
    name="agente_prueba",
    tools=[ejecutar_variante_simulada],
    hooks=[SoloSandboxHook(), AuditoriaHook()],
    callback_handler=None,
)

respuesta = agente_prueba.tool.ejecutar_variante_simulada(
    host="prod-web-01", proceso="python3", proceso_padre="bash", puerto_destino=443
)
print("Resultado:", respuesta["status"], "→", respuesta["content"][0]["text"])
print("¿Se escribió algún evento nuevo en los logs?", len(LOGS) > len(LINEA_BASE))

---

## Parte 3: Los agentes del equipo rojo, el equipo azul y el revisor

Cada agente tiene un **rol**, unas **herramientas** y unas **reglas de juego** en su `system_prompt`. Fíjate en tres detalles que vienen directamente del artículo:

- Al equipo rojo se le pide **reintentar** si una ejecución falla (máximo 3 intentos).
- A los dos equipos se les prohíbe **inventar resultados**: solo cuenta lo que devuelven las herramientas.
- El revisor **no despliega nada**: redacta una solicitud de cambio para una persona.

In [ ]:
HOOKS_DE_SEGURIDAD = [SoloSandboxHook(), AuditoriaHook()]

equipo_rojo = Agent(
    name="equipo_rojo",
    system_prompt="""Eres el EQUIPO ROJO de un sistema autónomo de pruebas de seguridad.
Trabajas SOLO en un entorno aislado y simulado con estos hosts: sandbox-01, sandbox-02, sandbox-03.
Tu misión es comprobar si la regla de detección vigente tiene huecos generando variantes de la
técnica "canal de control remoto desde un intérprete" con la herramienta ejecutar_variante_simulada.

Instrucciones:
- Ejecuta al menos 6 variantes DISTINTAS: cambia el intérprete (python, python3, python3.11, python3.12),
  el proceso padre (bash, cron, nginx, systemd) y el puerto de destino (4444, 443, 8080, 53).
- Si una ejecución falla, lee el error, corrige el parámetro y reinténtalo (máximo 3 intentos por variante).
- No inventes resultados: solo cuentan las variantes con un ID de evidencia (EVT-XXXX) devuelto por la herramienta.
- Termina con una lista breve: una línea por variante con su ID de evidencia.""",
    tools=[ejecutar_variante_simulada],
    hooks=HOOKS_DE_SEGURIDAD,
    callback_handler=None,
)

equipo_azul = Agent(
    name="equipo_azul",
    system_prompt=f"""Eres el EQUIPO AZUL de un sistema autónomo de pruebas de seguridad.
Tu misión es que la regla de detección cubra todas las variantes del equipo rojo SIN lanzar
falsas alarmas sobre la actividad legítima (eventos con origen "linea_base").

Regla vigente:
{json.dumps(REGLA_ACTUAL, indent=2)}

Pasos:
1. Usa consultar_logs y compara los eventos con origen "equipo_rojo" frente a "linea_base".
2. Evalúa la regla vigente con evaluar_regla para medir su hueco.
3. Propón una regla mejorada (con un nombre nuevo, por ejemplo canal-remoto-python-v2) y evalúala.
   Itera hasta lograr precisión 1.0 y recall 1.0 (máximo 4 intentos).
4. Responde con el hueco que encontraste, la regla final (todos sus parámetros) y sus métricas.
Las métricas solo valen si las devolvió evaluar_regla; nunca las estimes tú.""",
    tools=[consultar_logs, evaluar_regla],
    hooks=HOOKS_DE_SEGURIDAD,
    callback_handler=None,
)

revisor = Agent(
    name="revisor",
    system_prompt="""Eres el REVISOR. Redacta en español, en menos de 150 palabras y en formato Markdown,
una solicitud de cambio para que un analista humano apruebe o rechace el despliegue de la nueva
regla en producción. Incluye: el hueco encontrado en la regla vigente, la regla propuesta,
sus métricas y los riesgos o próximos pasos. No despliegues nada: la decisión final es humana.""",
    callback_handler=None,
)

print("✅ Agentes creados: equipo_rojo, equipo_azul, revisor")

---

## Parte 4: El grafo que los coordina

Con `GraphBuilder` definimos **nodos** (los agentes) y **aristas** (quién va después de quién). Las aristas condicionales crean el **ciclo competitivo** de ATA:

- `equipo_rojo → equipo_azul`: siempre.
- `equipo_azul → revisor`: solo si la regla está **validada** (precisión y recall = 1.0).
- `equipo_azul → equipo_rojo`: si la regla todavía falla, empieza una nueva ronda.

**Lo importante:** las condiciones (`regla_validada`) son funciones de Python que vuelven a medir la regla con los logs. **El grafo no decide según lo que "diga" el LLM, sino según la evidencia.** Esto es la ejecución fundamentada aplicada al flujo.

Como el grafo tiene un ciclo, ponemos **límites de seguridad**: un número máximo de ejecuciones de nodos y un tiempo máximo.

In [ ]:
from strands.multiagent import GraphBuilder

UMBRAL = 1.0


def regla_validada(state) -> bool:
    """Decisión fundamentada: se mide con los logs reales, no con lo que diga el LLM."""
    mejor = mejor_regla()
    return mejor is not None and mejor["precision"] >= UMBRAL and mejor["recall"] >= UMBRAL


def regla_no_validada(state) -> bool:
    return not regla_validada(state)


builder = GraphBuilder()
builder.add_node(equipo_rojo, "equipo_rojo")
builder.add_node(equipo_azul, "equipo_azul")
builder.add_node(revisor, "revisor")

builder.add_edge("equipo_rojo", "equipo_azul")
builder.add_edge("equipo_azul", "revisor", condition=regla_validada)
builder.add_edge("equipo_azul", "equipo_rojo", condition=regla_no_validada)  # nueva ronda

builder.set_entry_point("equipo_rojo")  # el ciclo da a todos los nodos una arista de entrada
builder.set_max_node_executions(7)      # como máximo 3 rondas rojo ↔ azul + revisor
builder.set_execution_timeout(900)      # 15 minutos
builder.reset_on_revisit(True)          # cada ronda empieza con el agente "limpio"

grafo = builder.build()
print("✅ Grafo construido")

### ¡A ejecutar!

Tarda uno o dos minutos. Verás en directo las líneas `[ROJO]` y `[AZUL]` que imprimen las herramientas:

- El equipo rojo lanza **varias variantes a la vez**: Strands ejecuta en paralelo las herramientas que el modelo pide en un mismo turno, igual que ATA ejecuta de 10 a 30 variaciones concurrentes.
- Las líneas ⚠️ son intentos fallidos que el agente **corrige por sí solo** en el siguiente turno.

In [ ]:
resultado = grafo("Ejecuta una ronda de Análisis Autónomo de Amenazas sobre la regla 'canal-remoto-python-v1'.")

print("\nEstado:", resultado.status)
print("Orden de ejecución:", " → ".join(nodo.node_id for nodo in resultado.execution_order))

In [ ]:
for nodo in ["equipo_rojo", "equipo_azul", "revisor"]:
    if nodo in resultado.results:
        display(Markdown(f"### 🧩 Nodo `{nodo}`\n\n{resultado.results[nodo].result}"))

### Todas las reglas que probó el equipo azul

Volvemos a medir cada regla con los logs finales: la vigente frente a las que propuso el equipo azul.

In [ ]:
filas = [
    "| Regla | Procesos | Puertos | Destino externo | Entrada conectada a la red | Precisión | Recall |",
    "|---|---|---|---|---|---|---|",
]
for regla in REGLAS_PROBADAS:
    m = calcular_metricas(regla)
    filas.append(
        f"| `{regla['nombre']}` | {', '.join(regla['procesos'])} | {regla['puertos'] or 'cualquiera'} "
        f"| {regla['requiere_destino_externo']} | {regla['requiere_stdin_a_socket']} | {m['precision']} | {m['recall']} |"
    )
display(Markdown("\n".join(filas)))

### Experimento: ¿por qué no basta con vigilar "cualquier Python que conecte hacia fuera"?

Una regla muy amplia captura todos los ataques (recall alto), pero también la actividad legítima de la línea base, como `pip install` o los scripts que llaman a una API. Mídela tú mismo (sin LLM):

In [ ]:
regla_amplia = {
    "nombre": "demasiado-amplia",
    "procesos": ["python", "python3", "python3.11", "python3.12"],
    "puertos": [],                       # cualquier puerto
    "requiere_destino_externo": True,
    "requiere_stdin_a_socket": False,    # ← sin la señal de comportamiento clave
}
print(calcular_metricas(regla_amplia))

Los falsos positivos bajan la precisión. Esta es la tensión que el equipo azul resuelve: **añadir la condición justa** (aquí, que la entrada del proceso esté conectada a la red) para no perder ataques ni molestar con falsas alarmas.

---

## Parte 5: Ejecución fundamentada — ¿hay pruebas?

En ATA, si un agente dice "ejecuté la técnica", **tiene que haber un log que lo demuestre**. Vamos a comprobarlo: extraemos los IDs de evidencia que menciona el equipo rojo y los buscamos en la base de logs.

También probamos qué pasa con una afirmación inventada (una "alucinación").

In [ ]:
import re


def verificar_evidencia(texto: str) -> None:
    """Acepta solo las afirmaciones respaldadas por un evento real del equipo rojo."""
    eventos = {e["id"]: e for e in LOGS}
    for evt_id in sorted(set(re.findall(r"EVT-\d{4}", texto))):
        evento = eventos.get(evt_id)
        if evento and evento["origen"] == "equipo_rojo":
            print(f"✅ {evt_id}: {evento['proceso']} en {evento['host']} a las {evento['timestamp']}")
        else:
            print(f"❌ {evt_id}: sin evidencia en los logs → afirmación descartada")


print("Afirmaciones del equipo rojo:")
verificar_evidencia(str(resultado.results["equipo_rojo"].result))

print("\nY si un agente 'alucina' una ejecución:")
verificar_evidencia("He ejecutado con éxito la variante EVT-9999.")

---

## Parte 6: Supervisión humana antes de producción

El artículo es claro: **una persona aprueba los cambios antes de desplegarlos en producción**. En Strands lo hacemos con una **interrupción**:

1. El agente de despliegue intenta llamar a `desplegar_regla_en_produccion`.
2. El hook `AprobacionHumanaHook` llama a `event.interrupt(...)` y el agente **se detiene** (`stop_reason == "interrupt"`).
3. Una persona revisa la propuesta y responde.
4. Reanudamos el agente con la respuesta. Si no es `"aprobar"`, el hook cancela la herramienta.

Además, la herramienta solo acepta la regla que **ha sido medida** como la mejor: otra salvaguarda fundamentada.

In [ ]:
REGLAS_EN_PRODUCCION: list[dict] = []


@tool
def desplegar_regla_en_produccion(nombre: str) -> str:
    """Despliega en producción una regla de detección ya validada.

    Args:
        nombre: Nombre de la regla que se va a desplegar
    """
    candidata = mejor_regla()
    if candidata is None or candidata["regla"]["nombre"] != nombre:
        return f"Error: '{nombre}' no es la regla validada. Solo se puede desplegar la mejor regla medida."
    REGLAS_EN_PRODUCCION.append(candidata["regla"])
    return f"Regla '{nombre}' desplegada en producción."


class AprobacionHumanaHook(HookProvider):
    """Detiene al agente antes de desplegar y espera la decisión de una persona."""

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeToolCallEvent, self.pedir_aprobacion)

    def pedir_aprobacion(self, event: BeforeToolCallEvent) -> None:
        if event.tool_use["name"] != "desplegar_regla_en_produccion":
            return
        # La primera vez detiene al agente; al reanudar, devuelve la respuesta humana
        decision = event.interrupt("aprobacion_despliegue", reason=mejor_regla())
        if decision != "aprobar":
            event.cancel_tool = "Despliegue rechazado por el revisor humano."


agente_despliegue = Agent(
    name="agente_despliegue",
    system_prompt="Eres el agente de despliegue. Despliega en producción la regla que te indiquen usando tu herramienta.",
    tools=[desplegar_regla_en_produccion],
    hooks=[AprobacionHumanaHook(), AuditoriaHook()],
    callback_handler=None,
)

assert mejor_regla() is not None, "Ejecuta primero el grafo de la Parte 4"
nombre_regla = mejor_regla()["regla"]["nombre"]
resultado_despliegue = agente_despliegue(f"Despliega la regla '{nombre_regla}' en producción.")

print("Motivo de parada:", resultado_despliegue.stop_reason)
for interrupcion in resultado_despliegue.interrupts or []:
    print(f"\n⏸️  Interrupción '{interrupcion.name}' — propuesta para revisar:")
    print(json.dumps(interrupcion.reason, indent=2, ensure_ascii=False))

### 👤 Tu turno: eres el analista humano

Revisa la propuesta de arriba (y la solicitud de cambio del revisor en la Parte 4). Cambia `DECISION_HUMANA` a `"rechazar"` si quieres ver qué pasa cuando no apruebas.

In [ ]:
DECISION_HUMANA = "aprobar"   # o "rechazar"

respuestas = [
    {"interruptResponse": {"interruptId": interrupcion.id, "response": DECISION_HUMANA}}
    for interrupcion in resultado_despliegue.interrupts
]
resultado_despliegue = agente_despliegue(respuestas)

display(Markdown(str(resultado_despliegue)))
print("Reglas en producción:", [regla["nombre"] for regla in REGLAS_EN_PRODUCCION] or "ninguna")

---

## Parte 7: Registro de auditoría

Gracias a `AuditoriaHook`, cada acción de cada agente quedó registrada, incluida la llamada bloqueada por la salvaguarda. Si el equipo rojo eligió un intérprete que no existía en algún host, verás filas con estado `error` seguidas del reintento corregido. En un sistema real este registro iría a un almacén inmutable (por ejemplo, CloudWatch Logs o S3 con bloqueo de objetos).

In [ ]:
filas = ["| Hora (UTC) | Agente | Herramienta | Host | Estado |", "|---|---|---|---|---|"]
for registro in AUDITORIA:
    host = registro["entrada"].get("host", "—")
    filas.append(f"| {registro['hora']} | {registro['agente']} | `{registro['herramienta']}` | {host} | {registro['estado']} |")
display(Markdown("\n".join(filas)))

---

## Resumen

| Lo que hiciste | Pieza de Strands | Idea de ATA |
|---|---|---|
| Simulaste ataques en un sandbox y dejaste evidencia | `@tool` + base de logs | Ejecución fundamentada |
| Mejoraste una regla midiendo precisión y recall | `Agent` con herramientas de medición | Equipo azul |
| Coordinaste rojo ↔ azul hasta validar la regla | `GraphBuilder`, aristas condicionales, límites | Ciclo competitivo en grafo |
| Bloqueaste hosts fuera del sandbox | Hook `BeforeToolCallEvent` + `cancel_tool` | Entorno aislado y control de acceso |
| Registraste cada acción | Hook `AfterToolCallEvent` | Auditoría completa |
| Detuviste el despliegue hasta tener aprobación | `event.interrupt` + reanudación | Supervisión humana |

**La idea central:** los agentes aportan velocidad y creatividad, pero **las decisiones importantes se apoyan en evidencia medible y en la aprobación de una persona**. La IA aumenta al equipo de seguridad, no lo sustituye.

---

## 🎯 Pruébalo tú

1. **Rompe la precisión.** Añade a `LINEA_BASE` un evento legítimo que se parezca al ataque (por ejemplo, una herramienta interna en `python3` con `stdin_a_socket=True` hacia un destino interno). Vuelve a ejecutar desde la Parte 1. ¿Qué condición tiene que usar ahora el equipo azul?
2. **Limita el ciclo.** Cambia `set_max_node_executions(7)` por `2`. ¿Qué estado devuelve el grafo si la regla no llega a validarse a tiempo?
3. **Rechaza el despliegue.** Pon `DECISION_HUMANA = "rechazar"` y revisa la tabla de auditoría.
4. **Escala en paralelo.** El artículo habla de 10 a 30 variaciones a la vez. Añade un segundo nodo de equipo rojo como otro punto de entrada que también alimente a `equipo_azul`. Pista: en Python el grafo usa semántica OR, así que revisa en la documentación cómo [esperar a todas las dependencias](https://strandsagents.com/docs/user-guide/sdk/multi-agent/graph/#waiting-for-all-dependencies).

## 📚 Para saber más

- [Graph (multi-agente)](https://strandsagents.com/docs/user-guide/sdk/multi-agent/graph/)
- [Hooks](https://strandsagents.com/docs/user-guide/sdk/agents/hooks/)
- [Hook events](https://strandsagents.com/docs/user-guide/sdk/agents/hooks-events/)
- [Responsible AI en Strands](https://strandsagents.com/docs/user-guide/sdk/safety-security/responsible-ai/)